# Enzyme Kinetics II --- Instructor Solutions

This is the **instructor answer key** for the three exercises in
[`notebooks/04_enzyme_kinetics_II.ipynb`](../notebooks/04_enzyme_kinetics_II.ipynb). For each exercise it gives:

1. a full derivation or worked reasoning (not just the answer),
2. the completed code, runnable end-to-end, and
3. **teaching notes**: what a fully-correct submission should show, common
   student mistakes, and a talking point/extension if there's time.

**Not for distribution to students before the exercise deadline.** This notebook
is excluded from the published Quarto site (see `_quarto.yml`'s `!instructor/`
render rule) but is still committed to the repo history like any other file.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

Vmax, KM, Ki = 10.0, 5.0, 2.0
S = np.linspace(0, 50, 400)

def v_competitive(S, I):
    KM_app = KM * (1 + I / Ki)
    return Vmax * S / (KM_app + S)

def v_noncompetitive(S, I):
    Vmax_app = Vmax / (1 + I / Ki)
    return Vmax_app * S / (KM + S)

def hill_activation(S, n, K=KM):
    return Vmax * S**n / (K**n + S**n)

n_values = [1, 2, 4, 8]


## Exercise 1: Inhibition at $S=K_M$, $I=K_i$

### Derivation

At $S=K_M$, $I=0$: $v_0=V_{max}\cdot K_M/(2K_M)=V_{max}/2$ — the usual baseline.

**Non-competitive** at $I=K_i$: $V_{max}^{app}=V_{max}/(1+I/K_i)=V_{max}/2$, and the
$S/(K_M+S)$ factor is *untouched* by non-competitive inhibition (it doesn't change
$K_M$). So

$$
\frac{v_{noncomp}(K_M, K_i)}{v_0(K_M,0)} = \frac{V_{max}^{app}}{V_{max}} = \frac{1}{1+I/K_i} = \frac12
$$

**at every $[S]$, not just $S=K_M$** — that ratio-independent-of-$S$ property is the
defining signature of non-competitive inhibition.

**Competitive** at $I=K_i$: $K_M^{app}=K_M(1+I/K_i)=2K_M$, so

$$
v_{comp}(K_M,K_i) = \frac{V_{max}K_M}{2K_M+K_M} = \frac{V_{max}}{3},
$$

giving $v_{comp}/v_0 = (V_{max}/3)/(V_{max}/2) = 2/3$ — **less** inhibition (33%,
vs. 50% for non-competitive) at this particular $[S]$, but — unlike non-competitive
— this fraction *does* depend on $[S]$: competitive inhibition can always be
"outcompeted" by raising $[S]$ enough (the ratio $\to1$ as $S\to\infty$), while
non-competitive inhibition cannot (the ratio stays $1/(1+I/K_i)$ at every $S$,
including $S\to\infty$).


In [2]:
v0_baseline = v_competitive(KM, 0)   # = Vmax/2, either function at I=0 gives the same thing
v_comp = v_competitive(KM, Ki)
v_noncomp = v_noncompetitive(KM, Ki)

print(f"v0 baseline (S=KM, I=0):        {v0_baseline:.4f}")
print(f"Competitive (S=KM, I=Ki):        {v_comp:.4f}  ({100 * v_comp / v0_baseline:.1f}% of baseline)")
print(f"Non-competitive (S=KM, I=Ki):    {v_noncomp:.4f}  ({100 * v_noncomp / v0_baseline:.1f}% of baseline)")

# Confirm the non-competitive ratio holds at EVERY S, not just S=KM
ratio_noncomp = v_noncompetitive(S[1:], Ki) / v_competitive(S[1:], 0)
print(f"Non-competitive/baseline ratio range across all S: "
      f"[{ratio_noncomp.min():.4f}, {ratio_noncomp.max():.4f}] (should be a single constant value)")


v0 baseline (S=KM, I=0):        5.0000
Competitive (S=KM, I=Ki):        3.3333  (66.7% of baseline)
Non-competitive (S=KM, I=Ki):    2.5000  (50.0% of baseline)
Non-competitive/baseline ratio range across all S: [0.5000, 0.5000] (should be a single constant value)


### Teaching notes

**What a correct submission looks like:** ≈66.7% for competitive, exactly 50% for
non-competitive, plus the (often-missed) observation that the non-competitive
fraction is constant across *all* $[S]$ while the competitive fraction is not.

**Common mistakes:**

- Computing both ratios only at $S=K_M$ and concluding "competitive inhibits less
  than non-competitive" as a blanket statement — true at this particular $[S]$, but
  the competitive fraction *changes* with $[S]$ (approaching no inhibition at high
  $[S]$), so the comparison is $[S]$-dependent, not universal.
- Mixing up which inhibition type keeps $K_M$ fixed vs. $V_{max}$ fixed — a common
  fix is to say the name out loud: "competitive" competes for the *same
  binding site*, raising apparent $K_M$; "non-competitive" doesn't affect binding,
  it caps the maximum rate.

**Extension, if there's time:** ask students to find the $[S]$ at which competitive
inhibition's effect has shrunk to, say, 10% loss of activity, given fixed $I=K_i$ —
this is the practical "how much substrate do I need to add to overcome this
inhibitor" question that motivates why competitive inhibitors are dose-dependent in
a way non-competitive ones aren't.


## Exercise 2: The Half-Max Point Is Always $K$

### Derivation

Setting $\text{hill\_activation}(S,n)=V_{max}/2$:

$$
\frac{V_{max}S^n}{K^n+S^n}=\frac{V_{max}}{2}
\;\Longrightarrow\; 2S^n = K^n+S^n
\;\Longrightarrow\; S^n=K^n
\;\Longrightarrow\; S=K,
$$

for **any** $n>0$ — the half-max point of a Hill function is always exactly $K$,
independent of the cooperativity. Only the *steepness* of the transition around
that point depends on $n$, not its location. Solved numerically below via
`brentq` (root-finding, rather than reading an approximate crossing off the plotted
grid) to make the "exactly $K$, for every $n$" claim precise.


In [3]:
print(f"K = {KM}\n")
for n in n_values:
    S_half = brentq(lambda s: hill_activation(s, n) - Vmax / 2, 1e-6, 100)
    print(f"n={n}: S at half-max = {S_half:.6f}  (K = {KM})")


K = 5.0

n=1: S at half-max = 5.000000  (K = 5.0)
n=2: S at half-max = 5.000000  (K = 5.0)
n=4: S at half-max = 5.000000  (K = 5.0)
n=8: S at half-max = 5.000000  (K = 5.0)


### Teaching notes

**What a correct submission looks like:** the solved `S_half` equals `KM` to
solver tolerance for every $n$ tested — the whole point being that this *doesn't*
change with $n$, even though the curve shape changes dramatically.

**Common mistakes:**

- Estimating $S_{1/2}$ visually from the plotted `S` grid instead of solving for it,
  which can look "close to $K$ but maybe drifting with $n$" purely from plotting
  resolution — the root-finder removes that ambiguity.
- Confusing "the half-max point is fixed at $K$" with "the curve looks the same for
  every $n$" — it very much doesn't (that's the entire content of Exercise 3); only
  the crossing point is invariant.

**Extension, if there's time:** ask what value of $K$ would be needed to move the
half-max point to a specific target $[S]$ for a *fixed* $n$ — reframes $K$ as the
tunable "set point" of a switch, independent of how sharp that switch is.


## Exercise 3: Ultrasensitivity — the 10%–90% Range

### Derivation

Solving $\text{hill\_activation}(S,n)=pV_{max}$ for $S$ gives

$$
S = K\left(\frac{p}{1-p}\right)^{1/n}.
$$

So $S_{10}=K(1/9)^{1/n}$ and $S_{90}=K\cdot9^{1/n}$, giving

$$
\frac{S_{90}}{S_{10}} = 9^{2/n} = 81^{1/n}.
$$

For $n=1$ (ordinary Michaelis-Menten shape) this ratio is $81$ — an 81-fold range in
$[S]$ is needed to go from 10% to 90% activation. For $n=8$, $81^{1/8}\approx1.73$ —
under a 2-fold range. This 81-fold-vs-2-fold contrast **is** the quantitative
definition of ultrasensitivity: a high Hill coefficient compresses the
"partially-on" range into a narrow band around $K$, producing switch-like behavior.


In [4]:
print(f"{'n':>3} | {'S_10 (analytic)':>16} | {'S_90 (analytic)':>16} | {'S90/S10':>10}")
for n in n_values:
    S10_analytic = KM * (0.1 / 0.9) ** (1 / n)
    S90_analytic = KM * (0.9 / 0.1) ** (1 / n)
    # cross-check numerically via root-finding
    S10_numeric = brentq(lambda s: hill_activation(s, n) - 0.1 * Vmax, 1e-6, 1000)
    S90_numeric = brentq(lambda s: hill_activation(s, n) - 0.9 * Vmax, 1e-6, 1000)
    assert abs(S10_numeric - S10_analytic) < 1e-6
    assert abs(S90_numeric - S90_analytic) < 1e-6
    ratio = S90_analytic / S10_analytic
    print(f"{n:>3} | {S10_analytic:>16.4f} | {S90_analytic:>16.4f} | {ratio:>10.3f}")


  n |  S_10 (analytic) |  S_90 (analytic) |    S90/S10
  1 |           0.5556 |          45.0000 |     81.000
  2 |           1.6667 |          15.0000 |      9.000
  4 |           2.8868 |           8.6603 |      3.000
  8 |           3.7992 |           6.5804 |      1.732


### Teaching notes

**What a correct submission looks like:** $S_{90}/S_{10}=81$ for $n=1$ and $\approx
1.73$ for $n=8$ (and the general-$n$ table shows this ratio $=81^{1/n}$ shrinking
monotonically as $n$ grows), with analytic and numerically-solved values agreeing to
solver tolerance.

**Common mistakes:**

- Reporting $S_{90}-S_{10}$ (an absolute difference) instead of the *ratio*
  $S_{90}/S_{10}$ — the ratio is the standard, dimensionless "sharpness" measure
  used in the ultrasensitivity literature and is what makes the $81^{1/n}$ formula
  clean; the absolute difference doesn't have as clean a closed form and is harder
  to compare across different $K$ values.
- Solving only for $n=1$ and $n=8$ as asked, but missing the more general pattern —
  worth having them fill in $n=2,4$ too (as done above) to *see* the monotonic
  compression rather than just two isolated data points.

**Extension, if there's time:** ask what $n$ would be needed for $S_{90}/S_{10}\le2$
(a genuinely switch-like response) — solving $81^{1/n}=2$ gives $n=\ln81/\ln2\approx
6.34$, i.e. $n\gtrsim7$ (matching the $n=8$ curve already being close to 2-fold
above) is roughly the threshold most texts call "highly cooperative."


## Grading rubric summary

| Exercise | Full marks requires | Partial credit for |
|---|---|---|
| 1. Inhibition at $S=K_M$ | Both ratios computed correctly (66.7% competitive, 50% non-competitive); explanation of the $[S]$-(in)dependence contrast | Correct numbers, no discussion of $[S]$-dependence |
| 2. Half-max = $K$ | Root-solved (not eyeballed) $S_{1/2}=K$ for every $n$ tested | Correct claim, verified only by inspection of the plot |
| 3. Ultrasensitivity ratio | Correct $81^{1/n}$ formula; $n=1$ vs $n=8$ contrast (81 vs. ~1.73) stated with the "switch-like" interpretation | Correct numeric ratios, no connection to ultrasensitivity/switch behavior |

**Reference:** Ingalls, B. P. (2013). *Mathematical Modeling in Systems Biology: An
Introduction*. MIT Press. Official PDF (with solutions):
<https://www.math.uwaterloo.ca/~bingalls/MMSB/MMSB_w_solutions.pdf>
